# DialReader: End-to-End Mechanical Dial Reader with ANN Digit Classifier

This notebook walks through the complete Machine Learning pipeline for reading 4-digit mechanical counter dials from photos using a **Two-Stage Architecture**:

1. **Stage 1: Display Window Detector (YOLO)** — Locates the glossy rectangular window on the orange dial body.
2. **Normalization & Crop Splitter** — Rectifies orientation and extracts 4 discrete digit drum slots.
3. **Stage 2: Digit Classifier (ANN / Multi-Layer Perceptron)** — An Artificial Neural Network that classifies ONE digit slot crop (0–9) at a time, evaluated 4 times per dial.

All logic is imported directly from modular packages in `src/` to ensure zero code duplication.

--- 
### 1. Environment Setup & Modular Imports
We import our core modules from `src/` and verify the PyTorch installation.

In [1]:
import sys
from pathlib import Path

# Ensure dial-reader root is on sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Import core modules from src/
from src.models import DigitANN, DigitANNPredictor, get_ann_transform
from src.train_digits import load_config, train_digit_ann
from src.crop import WindowNormalizer
from src.detect import DialDetector
from src.infer import DialReadingPipeline
from src.eval import evaluate_test_set
from src.run_experiments import run_experiments
from src.utils.generate_digit_crops import generate_digit_crops

print("✓ All DialReader modules successfully imported without errors!")
print(f"✓ PyTorch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

✓ All DialReader modules successfully imported without errors!
✓ PyTorch version: 2.14.1+cpu | CUDA available: False


--- 
### 2. Dataset Inspection & Labeling Status
We inspect `data/labeled/labels.csv` to track progress:
- **Knife dataset** (photos + sparse video frames) = Train / Validation pool.
- **Knife+Pressing dataset** = Held-out Test set.
- **Images 116/117** = Negative rejection controls.

In [2]:
labels_csv = PROJECT_ROOT / "data" / "labeled" / "labels.csv"
raw_dir = PROJECT_ROOT / "data" / "raw"

if labels_csv.exists():
    df = pd.read_csv(labels_csv)
    total_count = len(df)
    labeled_mask = (df["reading"].notna()) & (df["reading"].astype(str).str.len() == 4) & (df["bbox"].notna())
    labeled_count = int(labeled_mask.sum())
    
    print(f"Total cataloged dial images: {total_count}")
    print(f"Fully labeled windows: {labeled_count} ({labeled_count / max(1, total_count) * 100:.1f}%)")
    display(df[labeled_mask].head() if labeled_count > 0 else df.head())
else:
    print(f"labels.csv not found at {labels_csv}.")

Total cataloged dial images: 152
Fully labeled windows: 0 (0.0%)


,filename,bbox,reading,orientation,mid_roll,dial_variant,split_group
0,knife/20261006_083755899_iOS.jpg,NaN,NaN,0,False,standard_3black_1red,train_or_val
1,knife/20261006_083758932_iOS.jpg,NaN,NaN,0,False,standard_3black_1red,train_or_val
2,knife/20261006_083809306_iOS.jpg,NaN,NaN,0,False,standard_3black_1red,train_or_val
3,knife/20261006_083814174_iOS.jpg,NaN,NaN,0,False,standard_3black_1red,train_or_val
4,knife/20261006_083827117_iOS.jpg,NaN,NaN,0,False,standard_3black_1red,train_or_val


--- 
### 3. Crop Generation & Visual Inspection
Before training the ANN, labeled display windows are extracted into 4 discrete slot crops and organized into:
`data/digits/{train, val, test}/{0..9}/`

The partition is strictly deterministic by source (**Knife** vs **Knife+Pressing**), avoiding random data leakage.

In [3]:
# Generate / refresh digit crops from labeled entries
digits_dir = PROJECT_ROOT / "data" / "digits"
crop_summary = generate_digit_crops(
    labels_csv=labels_csv,
    raw_dir=raw_dir,
    output_dir=digits_dir
)

# Display sample crops if available
sample_crops = list(digits_dir.glob("*/*/*.png"))
if sample_crops:
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        if i < len(sample_crops):
            img = Image.open(sample_crops[i])
            ax.imshow(img, cmap="gray" if img.mode == "L" else None)
            ax.set_title(f"Digit: {sample_crops[i].parent.name}")
        ax.axis("off")
    plt.suptitle("Sample Extracted Digit Slot Crops", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No digit crops found on disk yet. Complete labeling first.")

[generate_digit_crops] Processing 152 entries from C:\Users\ABHI\OneDrive\Desktop\DialReader\dial-reader\data\labeled\labels.csv...

DIGIT CROP EXTRACTION SUMMARY
Total labeled windows parsed: 0
Negative samples skipped (116/117): 6
Total single digit crops saved: 0
--------------------------------------------------
Digit | Train |  Val  | Test | Total
--------------------------------------------------
  0   |     0 |     0 |    0 |     0
  1   |     0 |     0 |    0 |     0
  2   |     0 |     0 |    0 |     0
  3   |     0 |     0 |    0 |     0
  4   |     0 |     0 |    0 |     0
  5   |     0 |     0 |    0 |     0
  6   |     0 |     0 |    0 |     0
  7   |     0 |     0 |    0 |     0
  8   |     0 |     0 |    0 |     0
  9   |     0 |     0 |    0 |     0

No digit crops found on disk yet. Complete labeling first.


--- 
### 4. ANN Architecture & Parameter Count
Our reader model is an Artificial Neural Network (Multi-Layer Perceptron):
- **Input**: A flattened digit crop (64 height × 32 width × 1 channel = **2,048** input features).
- **Hidden Layers**: Dense linear layers with Batch Normalization, ReLU activation, and Dropout.
- **Output**: 10 linear units representing class logits for digits **0 through 9**.

All architecture parameters are configurable via `configs/digit_ann.yaml` without touching Python code.

In [4]:
# Load YAML configuration and inspect model capacity
cfg = load_config(PROJECT_ROOT / "configs" / "digit_ann.yaml")

input_h, input_w = cfg["data"]["input_size"]
channels = cfg["data"]["channels"]
input_dim = input_h * input_w * channels

model = DigitANN(
    input_dim=input_dim,
    hidden_layers=cfg["model"]["hidden_layers"],
    num_classes=cfg["model"]["num_classes"],
    activation=cfg["model"]["activation"],
    batchnorm=cfg["model"]["batchnorm"],
    dropout=cfg["model"]["dropout"]
)

params_info = model.count_parameters()

print(f"Input Dimension: {input_h} × {input_w} × {channels} = {input_dim:,} features")
print(f"Hidden Layers: {cfg['model']['hidden_layers']}")
print(f"Activation: {cfg['model']['activation']} | BatchNorm: {cfg['model']['batchnorm']} | Dropout: {cfg['model']['dropout']}")
print(f"Total Trainable Parameters: {params_info['trainable_parameters']:,}")
print("\nModel Architecture:\n", model)

[Config] Loaded settings from: C:\Users\ABHI\OneDrive\Desktop\DialReader\dial-reader\configs\digit_ann.yaml
Input Dimension: 64 × 32 × 1 = 2,048 features
Hidden Layers: [256, 128, 64]
Activation: relu | BatchNorm: True | Dropout: 0.2
Total Trainable Parameters: 567,242

Model Architecture:
 DigitANN(
  (feature_extractor): Sequential(
    (0): Linear(in_features=2048, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Dr

--- 
### 5. Train the ANN Digit Classifier *(Run After Labeling)*

> **Status**: Run this cell once you have labeled your images in `label_ui.py`.

Training includes:
- Class-weighted CrossEntropyLoss with Label Smoothing (epsilon = 0.05).
- AdamW optimizer with Cosine Annealing learning rate schedule.
- Early stopping to prevent overfitting on small real datasets.

In [5]:
# Check if training data exists before launching training
train_crops = list((PROJECT_ROOT / "data" / "digits" / "train").glob("*/*.png"))

if len(train_crops) > 0:
    print(f"Found {len(train_crops)} training crops. Launching ANN training...")
    train_results = train_digit_ann(
        config_path=PROJECT_ROOT / "configs" / "digit_ann.yaml",
        verbose=True
    )
else:
    print("[Notice] Labeled digit crops not found yet.")
    print("To train the model:")
    print("  1. Run the labeling UI: streamlit run label_ui.py")
    print("  2. Extract crops: python src/utils/generate_digit_crops.py")
    print("  3. Re-run this cell to train the ANN!")

[Notice] Labeled digit crops not found yet.
To train the model:
  1. Run the labeling UI: streamlit run label_ui.py
  2. Extract crops: python src/utils/generate_digit_crops.py
  3. Re-run this cell to train the ANN!


--- 
### 6. Plot Learning Curves (Loss & Accuracy)
Visualizes training and validation progression across epochs to diagnose bias and variance.

In [6]:
if 'train_results' in locals() and "history" in train_results:
    hist = train_results["history"]
    epochs_ran = range(1, len(hist["train_loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curve
    ax1.plot(epochs_ran, hist["train_loss"], label="Train Loss", color="#e74c3c", lw=2)
    ax1.plot(epochs_ran, hist["val_loss"], label="Val Loss", color="#3498db", lw=2)
    ax1.set_title("Cross-Entropy Loss vs Epochs", fontsize=13)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy curve
    ax2.plot(epochs_ran, hist["train_acc"], label="Train Accuracy", color="#27ae60", lw=2)
    ax2.plot(epochs_ran, hist["val_acc"], label="Val Accuracy", color="#9b59b6", lw=2)
    ax2.set_title("Classification Accuracy vs Epochs", fontsize=13)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("Learning curves will display here automatically once training completes.")

Learning curves will display here automatically once training completes.


--- 
### 7. Systematic Architecture Experiments *(Run After Labeling)*
Compares 10 systematic architecture modifications (One-Factor-at-a-Time) against the baseline.
Results are logged to `results/experiments.csv`.

In [7]:
exp_csv = PROJECT_ROOT / "results" / "experiments.csv"

# If crops exist, you can run full training sweeps; otherwise inspect dry-run table
if exp_csv.exists():
    df_exp = pd.read_csv(exp_csv)
    print("Existing Experiment Comparison Table:")
    display(df_exp)
else:
    print("Running dry-run parameter verification sweep across 10 configurations...")
    df_exp = run_experiments(
        config_path=str(PROJECT_ROOT / "configs" / "digit_ann.yaml"),
        results_csv=str(exp_csv),
        dry_run=True
    )
    display(df_exp)

Existing Experiment Comparison Table:


,experiment_name,description,hidden_layers,activation,batchnorm,dropout,lr,param_count,train_acc,val_acc,val_loss,best_epoch,status
0,baseline,Reference baseline architecture,"[256, 128, 64]",relu,True,0.2,0.0010,567242,NaN,NaN,NaN,NaN,dry_run_verified
1,arch_wide,"Wider hidden capacity [512, 256, 128]","[512, 256, 128]",relu,True,0.2,0.0010,1216394,NaN,NaN,NaN,NaN,dry_run_verified
2,arch_deep,"Deeper network [256, 256, 128, 64]","[256, 256, 128, 64]",relu,True,0.2,0.0010,633546,NaN,NaN,NaN,NaN,dry_run_verified
3,arch_compact,"Lightweight 2-layer network [128, 64]","[128, 64]",relu,True,0.2,0.0010,271562,NaN,NaN,NaN,NaN,dry_run_verified
4,act_gelu,GELU activation function,"[256, 128, 64]",gelu,True,0.2,0.0010,567242,NaN,NaN,NaN,NaN,dry_run_verified
5,act_leaky_relu,LeakyReLU (slope 0.1) activation,"[256, 128, 64]",leaky_relu,True,0.2,0.0010,567242,NaN,NaN,NaN,NaN,dry_run_verified
6,dropout_none,No dropout (p=0.0),"[256, 128, 64]",relu,True,0.0,0.0010,567242,NaN,NaN,NaN,NaN,dry_run_verified
7,dropout_high,Strong regularization (p=0.4),"[256, 128, 64]",relu,True,0.4,0.0010,567242,NaN,NaN,NaN,NaN,dry_run_verified
8,no_batchnorm,Ablation without Batch Normalization,"[256, 128, 64]",relu,False,0.2,0.0010,566346,NaN,NaN,NaN,NaN,dry_run_verified
9,lr_half,Lower learning rate (0.0005),"[256, 128, 64]",relu,True,0.2,0.0005,567242,NaN,NaN,NaN,NaN,dry_run_verified


--- 
### 8. Held-Out Test Set Evaluation *(Run After Training)*
Evaluates the complete 2-stage pipeline on the held-out **Knife+Pressing** dataset:
- 4-digit exact match accuracy
- Per-slot accuracy (slots 1, 2, 3, 4)
- Last-digit (mid-roll) error rate
- Negative sample (116/117) rejection rate
- Saves the 20 worst failure cases with visual banner annotations

In [8]:
det_weights = PROJECT_ROOT / "models" / "detector_best.pt"
ann_weights = PROJECT_ROOT / "models" / "digit_ann_best.pt"

if det_weights.exists() and ann_weights.exists():
    eval_metrics = evaluate_test_set(
        labels_csv=labels_csv,
        raw_dir=raw_dir,
        detector_weights=det_weights,
        classifier_weights=ann_weights,
        config_path=str(PROJECT_ROOT / "configs" / "digit_ann.yaml"),
        output_dir=PROJECT_ROOT / "runs" / "eval"
    )
    
    cm_img = PROJECT_ROOT / "runs" / "eval" / "confusion_matrices.png"
    if cm_img.exists():
        plt.figure(figsize=(10, 8))
        plt.imshow(Image.open(cm_img))
        plt.axis("off")
        plt.title("Held-Out Test Set Confusion Matrices (Slots 1-4)")
        plt.show()
else:
    print("Models not trained yet. Run after training detector and ANN weights.")

Models not trained yet. Run after training detector and ANN weights.


--- 
### 9. Sample Inference Demo
Runs the full two-stage pipeline on an individual test photo:
`Image -> YOLO Crop -> Rectify & Split Slots -> 4x ANN Classifications -> JSON Result`

In [9]:
test_images = list((raw_dir / "knife_pressing").glob("*.jpg")) + list((raw_dir / "knife_pressing").glob("*.jpeg"))

if test_images and det_weights.exists() and ann_weights.exists():
    sample_img_path = test_images[0]
    pipeline = DialReadingPipeline(
        detector_weights=det_weights,
        classifier_weights=ann_weights,
        config_path=str(PROJECT_ROOT / "configs" / "digit_ann.yaml")
    )
    
    result = pipeline.read_image(sample_img_path)
    print(f"Image: {sample_img_path.name}")
    print(f"Predicted Reading: {result.get('value')}")
    print(f"Confidence: {result.get('confidence')} | Status: {result.get('status')}")
    print(f"Per-Digit Confidences: {result.get('per_digit_confidence')}")
    print(f"Mid-Roll Transition: {result.get('is_mid_roll')}")
else:
    print("Inference demo ready. Will run on sample images once model weights are saved.")

Inference demo ready. Will run on sample images once model weights are saved.
